# 🚀 Asynchronous Python (`asyncio`): The Ultimate Beginner-to-Advanced Masterclass
**Welcome to Async Python!** If you are exploring asynchronous programming for the first time, this notebook is built just for you.

Based on industry best practices (including DataCamp's Async Python guide), this masterclass explains **every single concept from first principles** using real-world analogies, step-by-step interactive code cells, and practical benchmarks.

---

## 📌 Table of Contents
1. 🍕 **The Restaurant Analogy:** Synchronous vs. Asynchronous
2. 🧱 **Core Building Blocks & How to Run Async Code:** `async def`, `await`, and `asyncio.run()` vs Jupyter `await`
3. ⏱️ **Non-Blocking Delays:** `time.sleep()` vs `asyncio.sleep()`
4. ⚡ **Concurrent Task Execution:** `create_task()`, `gather()`, and `as_completed()`
5. ⏳ **Timeouts & Exception Resilience:** `wait_for()` and `return_exceptions=True`
6. 🧵 **Bridging Sync & Async:** Running Blocking Code with `asyncio.to_thread()`
7. 🔄 **Async Context Managers & Generators:** `async with` and `async for`
8. 📥 **Producer-Consumer Queues:** `asyncio.Queue` for Async Workflows
9. 🏎️ **Real-World Benchmark:** Sync Requests vs Async Batch Fetching
10. ⚠️ **Top 5 Common Mistakes Beginners Make**

---

## 🍕 1. The Restaurant Analogy (Why Async Exists?)

Imagine you are a waiter in a pizza restaurant with **1 chef in the kitchen**:

### ❌ Synchronous Waiter (Blocking Code):
1. You take Table 1's order for a pizza.
2. You give the order to the chef.
3. **You stand still in front of the oven for 15 minutes** doing absolutely nothing until the pizza is baked.
4. You deliver the pizza to Table 1.
5. Only THEN do you move to Table 2 to take their order.
   * **Result:** Extremely slow, customers leave, CPU/Waiter is wasted waiting!

### ✅ Asynchronous Waiter (Non-Blocking Code / `asyncio`):
1. You take Table 1's order for a pizza and give it to the chef.
2. While the pizza bakes (I/O Waiting), **you immediately walk to Table 2, Table 3, and Table 4** to take their orders.
3. When the kitchen timer dings (Pizza is ready!), you pause what you are doing and serve the pizza.
   * **Result:** 1 waiter handles 50 tables effortlessly!

### 🔑 Key Takeaway:
- **Sync Code:** Waits for network requests, database queries, or file reads to finish before starting the next line of code.
- **Async Code:** Pauses a function while waiting for I/O (Input/Output) and works on another task in the meantime on a **single CPU thread**!

## 🧱 2. Core Building Blocks & Running Async Code

To write asynchronous code in Python, you need to understand 3 main components:

1. `async def`: Defines a **Coroutine** (an asynchronous function that can be paused and resumed).
2. `await`: Means **"pause me here and let Python do other work until this result is ready"**.
3. **Event Loop & `asyncio.run()`**: The manager that manages and executes coroutines.

---

### ⚙️ How to Execute Async Code (`asyncio.run` vs Jupyter `await`)

#### A. In Standalone Python Scripts (`.py` files):
When running a standalone `.py` file from terminal, Python does **NOT** have an active event loop running by default. You MUST start the event loop using `asyncio.run(main())`:

```python
import asyncio

async def main():
    print("Hello from Async Python!")
    await asyncio.sleep(1)

if __name__ == "__main__":
    # asyncio.run() creates a new event loop, runs main(), and closes the loop when finished!
    asyncio.run(main())
```

#### B. Inside Jupyter Notebooks (`.ipynb` files):
Jupyter Notebooks (and IPython) **already have an active event loop running** in the background! Calling `asyncio.run(main())` inside Jupyter will throw a `RuntimeError: This event loop is already running`.
Instead, in Jupyter, you can simply call **`await main()`** directly in any cell!

Let's test this concept in code below:

In [ ]:
import asyncio

# 1. Define a regular synchronous function
def sync_greeting():
    return "Hello from Synchronous Python!"

# 2. Define an asynchronous coroutine
async def async_greeting():
    await asyncio.sleep(0.5)
    return "Hello from Asynchronous Python!"

# Calling sync function gives the value immediately:
print("Sync Call Output:", sync_greeting())

# Calling async function WITHOUT await returns a Coroutine object (not the result yet):
coro_object = async_greeting()
print("Async Call WITHOUT await:", coro_object)

# In Jupyter Notebooks, we execute the coroutine directly using 'await':
result = await async_greeting()
print("Async Call WITH await:", result)

# NOTE FOR .py SCRIPTS:
# In a terminal script (my_script.py), you would run: asyncio.run(async_greeting())

## ⏱️ 3. Non-Blocking Delay: `time.sleep` vs `asyncio.sleep` 

- `time.sleep(2)`: **Freezes the entire Python process** (Blocks everything).
- `await asyncio.sleep(2)`: **Pauses only this coroutine** and yields control back to the Event Loop so other tasks can run!

Let's write a function that simulates downloading 3 files.

In [7]:
import time
import asyncio

# --- SYNCHRONOUS VERSION ---
def fetch_data_sync(task_id, delay):
    print(f"[SYNC] Task {task_id}: Started downloading...")
    time.sleep(delay)  # Blocking delay
    print(f"[SYNC] Task {task_id}: Finished in {delay} seconds!")

print("=== Running Synchronous Batch ===")
start_time = time.time()
fetch_data_sync(1, 2)
fetch_data_sync(2, 2)
fetch_data_sync(3, 2)
print(f"Total Sync Time: {time.time() - start_time:.2f} seconds (2 + 2 + 2 = 6s)\n")

# --- ASYNCHRONOUS VERSION ---
async def fetch_data_async(task_id, delay):
    print(f"[ASYNC] Task {task_id}: Started downloading...")
    await asyncio.sleep(delay)  # Non-blocking pause!
    print(f"[ASYNC] Task {task_id}: Finished in {delay} seconds!")

print("=== Running Asynchronous Batch Concurrently ===")
start_time = time.time()
# asyncio.gather runs all 3 coroutines at the same time!
await asyncio.gather(
    fetch_data_async(1, 2),
    fetch_data_async(2, 2),
    fetch_data_async(3, 2)
)
print(f"Total Async Time: {time.time() - start_time:.2f} seconds (Ran concurrently!)")

=== Running Synchronous Batch ===
[SYNC] Task 1: Started downloading...
[SYNC] Task 1: Finished in 2 seconds!
[SYNC] Task 2: Started downloading...
[SYNC] Task 2: Finished in 2 seconds!
[SYNC] Task 3: Started downloading...
[SYNC] Task 3: Finished in 2 seconds!
Total Sync Time: 6.02 seconds (2 + 2 + 2 = 6s)

=== Running Asynchronous Batch Concurrently ===
[ASYNC] Task 1: Started downloading...
[ASYNC] Task 2: Started downloading...
[ASYNC] Task 3: Started downloading...
[ASYNC] Task 1: Finished in 2 seconds!
[ASYNC] Task 2: Finished in 2 seconds!
[ASYNC] Task 3: Finished in 2 seconds!
Total Async Time: 2.00 seconds (Ran concurrently!)


## ⚡ 4. Concurrent Task Execution

Python provides three primary ways to manage multiple concurrent tasks:

### 1. `asyncio.create_task(coroutine)`
Schedules a coroutine to start running **immediately in the background** on the event loop.

### 2. `asyncio.gather(*coroutines)`
Runs multiple coroutines concurrently and waits until **all** of them finish, returning a list of results in order.

### 3. `asyncio.as_completed(coroutines)`
Yields results **as soon as each task completes**, regardless of the order they were started.

In [8]:
async def download_file(file_name, delay):
    await asyncio.sleep(delay)
    return f"{file_name} (took {delay}s)"

# --- 1. USING asyncio.gather --- 
print("=== Demonstration 1: asyncio.gather ===")
start = time.time()
results = await asyncio.gather(
    download_file("File A", 3),
    download_file("File B", 1),
    download_file("File C", 2)
)
print("Gather Results (preserves input order):", results)
print(f"Time taken: {time.time() - start:.2f}s\n")

# --- 2. USING asyncio.as_completed --- 
print("=== Demonstration 2: asyncio.as_completed ===")
start = time.time()
tasks = [
    download_file("File A", 3),
    download_file("File B", 1),
    download_file("File C", 2)
]

# Processes results as soon as each task finishes:
for completed_task in asyncio.as_completed(tasks):
    result = await completed_task
    print(f"⚡ Finished right now: {result}")
print(f"Time taken: {time.time() - start:.2f}s")

=== Demonstration 1: asyncio.gather ===
Gather Results (preserves input order): ['File A (took 3s)', 'File B (took 1s)', 'File C (took 2s)']
Time taken: 3.00s

=== Demonstration 2: asyncio.as_completed ===
⚡ Finished right now: File B (took 1s)
⚡ Finished right now: File C (took 2s)
⚡ Finished right now: File A (took 3s)
Time taken: 3.01s


## ⏳ 5. Timeouts & Exception Handling

### 1. `asyncio.wait_for(coro, timeout)`
Automatically cancels a slow task after $N$ seconds by raising `asyncio.TimeoutError`.

### 2. `asyncio.gather(..., return_exceptions=True)`
Prevents a single failing task from crashing the entire batch.

In [ ]:
# --- 1. TIMEOUT DEMONSTRATION ---
async def slow_database_query():
    print("🔍 Querying database...")
    await asyncio.sleep(5)  # Takes 5 seconds
    return "Database Result"

try:
    print("=== Testing Timeout (Limit: 2 seconds) ===")
    result = await asyncio.wait_for(slow_database_query(), timeout=2.0)
    print(result)
except asyncio.TimeoutError:
    print("⚠️ Timeout Error: Database took longer than 2 seconds! Cancelled.\n")

# --- 2. EXCEPTION HANDLING DEMONSTRATION ---
async def risky_task(task_id):
    await asyncio.sleep(1)
    if task_id == 2:
        raise ValueError(f"Task {task_id} failed due to network error!")
    return f"Task {task_id} Success"

print("=== Handling Errors with return_exceptions=True ===")
results = await asyncio.gather(
    risky_task(1),
    risky_task(2),
    risky_task(3),
    return_exceptions=True  # Keeps other tasks alive!
)

for i, res in enumerate(results, start=1):
    if isinstance(res, Exception):
        print(f"Result {i}: ❌ Caught Exception -> {res}")
    else:
        print(f"Result {i}: ✅ {res}")

## 🧵 6. Running Blocking / Sync Code in Async (`asyncio.to_thread`)

What if you need to call a third-party synchronous library (like `requests.get()`, heavy image processing with `Pillow`, or legacy CPU math) inside an async application?

If you call a synchronous blocking function directly, **it will freeze your entire async event loop**.

### Solution: `asyncio.to_thread(func, *args)`
Python 3.9+ introduced `asyncio.to_thread()`, which offloads the blocking synchronous function onto a **separate background OS thread**, allowing the main async event loop to keep running smoothly!

In [ ]:
import time
import asyncio

# Imagine this is a legacy synchronous blocking function (e.g. heavy image processing or requests.get)
def heavy_blocking_computation(name, delay):
    print(f"[Thread] {name}: Heavy blocking task started on background thread...")
    time.sleep(delay)  # Blocking sync delay
    print(f"[Thread] {name}: Heavy computation done!")
    return f"{name} Result"

async def main_async_flow():
    print("=== Offloading Blocking Functions to Separate Threads ===")
    start = time.time()
    
    # Run blocking functions in background OS threads concurrently:
    task1 = asyncio.to_thread(heavy_blocking_computation, "Task A", 2)
    task2 = asyncio.to_thread(heavy_blocking_computation, "Task B", 2)
    
    results = await asyncio.gather(task1, task2)
    print("Results:", results)
    print(f"Total execution time: {time.time() - start:.2f}s (Ran concurrently on threads!)")

await main_async_flow()

## 🔄 7. Async Context Managers (`async with`) & Generators (`async for`)

### 1. `async with` (Context Managers)
Safely opens and closes asynchronous resources (e.g. `aiohttp.ClientSession()` or database connections).

### 2. `async for` (Async Iterators / Generators)
Yields items asynchronously over time (e.g. streaming LLM tokens, reading data streams, paginated API responses).

In [ ]:
# --- DEMONSTRATION OF ASYNC GENERATOR (async for) ---
async def stream_llm_tokens(prompt):
    """Simulates an LLM streaming response token by token."""
    response_tokens = ["Multimodal ", "RAG ", "with ", "LangChain ", "& ", "Pinecone!"]
    for token in response_tokens:
        await asyncio.sleep(0.3)  # Simulate network latency between tokens
        yield token  # Yield token asynchronously

print("=== Streaming LLM Tokens with 'async for' ===")
print("AI Response: ", end="", flush=True)

# Consume async generator using 'async for':
async for token in stream_llm_tokens("Explain RAG"):
    print(token, end="", flush=True)
print("\n")

## 📥 8. Producer-Consumer Pattern with `asyncio.Queue` 

In real-world data pipelines (like document extraction or web scraping), you often have **Producers** putting tasks into a Queue, and **Worker Consumers** pulling tasks out of the Queue to process them concurrently.

In [ ]:
async def producer(queue, n_items):
    """Produces tasks and puts them in the queue."""
    for i in range(1, n_items + 1):
        item = f"Document_{i}.pdf"
        await queue.put(item)
        print(f"[Producer] Added {item} to queue")
        await asyncio.sleep(0.2)

async def worker(worker_id, queue):
    """Worker consumer that processes items from the queue."""
    while True:
        item = await queue.get()
        print(f"  [Worker {worker_id}] Processing {item}...")
        await asyncio.sleep(0.8)  # Simulate PDF parsing
        print(f"  [Worker {worker_id}] Finished {item}")
        queue.task_done()

async def run_pipeline():
    print("=== Starting Producer-Consumer Pipeline ===")
    queue = asyncio.Queue()
    
    # Start 2 worker consumers in background
    workers = [asyncio.create_task(worker(i, queue)) for i in range(1, 3)]
    
    # Produce 5 tasks
    await producer(queue, 5)
    
    # Wait until all items in the queue are processed
    await queue.join()
    
    # Cancel worker background tasks
    for w in workers:
        w.cancel()
    print("Pipeline complete!")

await run_pipeline()

## 🏎️ 9. Real-World Benchmark: Sync vs Async API Fetching

Let's test fetching **10 API endpoints** sequentially using `time.sleep` vs concurrently using `asyncio`.

In [ ]:
import asyncio
import time
import random

URLS = [f"https://api.example.com/data/{i}" for i in range(1, 11)]

# --- 1. SYNCHRONOUS BATCH (Sequential) ---
def fetch_url_sync(url):
    delay = random.uniform(0.5, 1.0)  # Simulate network latency
    time.sleep(delay)
    return f"Response from {url} ({delay:.2f}s)"

print("=== Starting 10 Sequential Synchronous Requests ===")
start_sync = time.time()
sync_results = [fetch_url_sync(url) for url in URLS]
sync_duration = time.time() - start_sync
print(f"✅ Sync Completed in {sync_duration:.2f} seconds.\n")

# --- 2. ASYNCHRONOUS BATCH (Concurrent) ---
async def fetch_url_async(url):
    delay = random.uniform(0.5, 1.0)  # Simulate non-blocking network latency
    await asyncio.sleep(delay)
    return f"Response from {url} ({delay:.2f}s)"

print("=== Starting 10 Concurrent Asynchronous Requests ===")
start_async = time.time()

# NOTE ON THE '*' (UNPACKING OPERATOR):
# asyncio.gather() expects comma-separated arguments: gather(task1, task2, task3)
# The list comprehension [fetch_url_async(url)...] creates a single List: [task1, task2, task3]
# The '*' operator unpacks the list elements so gather sees them as individual comma-separated arguments!
async_results = await asyncio.gather(*[fetch_url_async(url) for url in URLS])
async_duration = time.time() - start_async
print(f"⚡ Async Completed in {async_duration:.2f} seconds.\n")

speedup = sync_duration / async_duration if async_duration > 0 else 0
print(f"🚀 Async Python was {speedup:.1f}x FASTER than Synchronous Python!")

## ⚠️ 10. Top 5 Common Mistakes Beginners Make

### ❌ Mistake 1: Forgetting `await`
```python
# WRONG: Calling an async function without await returns a coroutine object, it DOES NOT run it!
result = fetch_data()

# RIGHT:
result = await fetch_data()
```

### ❌ Mistake 2: Using Blocking Functions inside Async Code
```python
# WRONG: time.sleep(5) or requests.get() freezes the entire Event Loop for everyone!
async def bad_function():
    time.sleep(5)

# RIGHT: Use non-blocking async functions or offload with asyncio.to_thread():
async def good_function():
    await asyncio.sleep(5)
    # or await asyncio.to_thread(blocking_sync_func)
```

### ❌ Mistake 3: Running `asyncio.run()` Inside Jupyter Notebooks
Jupyter Notebooks already run an Event Loop in the background!
```python
# WRONG in Jupyter (raises RuntimeError: This event loop is already running):
asyncio.run(main())

# RIGHT in Jupyter: Call await directly in any cell!
await main()

# RIGHT in standalone .py script:
# if __name__ == '__main__': asyncio.run(main())
```

### ❌ Mistake 4: Expecting Async to Speed Up Heavy CPU Math Loops
Async is for **I/O-bound tasks** (Network, Disk, Databases). For heavy CPU math (Image processing, ML model training), use **Multiprocessing** or `asyncio.to_thread()`.

### ❌ Mistake 5: `await`ing in a Loop Sequentially
```python
# WRONG (Runs sequentially, no speed benefit!):
for url in urls:
    res = await fetch(url)

# RIGHT (Fires all tasks concurrently):
results = await asyncio.gather(*[fetch(url) for url in urls])
```

---

🎉 **Congratulations!** You now understand Asynchronous Python from first principles! You are ready to build ultra-fast, concurrent AI applications, RAG pipelines, and web scrapers.